In [1]:
import sys
import json
import csv
import argparse
import random

In [ ]:
def coerce_int_if_numeric(x):
    if isinstance(x, str) and x.isdigit():
        try:
            return int(x)
        except ValueError:
            return x
    return x

def read_text_input(path):
    if path == "-":
        raw = sys.stdin.read()
        src = "STDIN"
    else:
        if not os.path.exists(path):
            sys.exit(f"Input file not found: {path}")
        with open(path, "r", encoding="utf-8") as f:
            raw = f.read()
        src = path
    if not raw.strip():
        sys.exit(f"Input is empty ({src}).")
    return raw

def parse_as_json_array(raw, *, loose=False):
    # First try strict JSON
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            # allow a single object by wrapping into a list
            return [obj]
        if not isinstance(obj, list):
            sys.exit("Top-level JSON must be an array (or a single object).")
        return obj
    except json.JSONDecodeError as e:
        if not loose:
            raise
        # Loose fallback via ast.literal_eval (handles single quotes/trailing commas)
        try:
            obj = ast.literal_eval(raw)
            if isinstance(obj, dict):
                return [obj]
            if not isinstance(obj, list):
                sys.exit("Loose parse succeeded, but top-level is not a list/dict.")
            return obj
        except Exception as e2:
            # Last-ditch attempt: if it looks like a single object, wrap it
            stripped = raw.strip()
            if stripped.startswith("{") and stripped.endswith("}"):
                try:
                    return [json.loads(stripped)]
                except Exception:
                    pass
            raise

def parse_as_json_lines(raw):
    records = []
    lines = [ln for ln in raw.splitlines() if ln.strip()]
    for idx, ln in enumerate(lines):
        try:
            obj = json.loads(ln)
        except json.JSONDecodeError as e:
            # Allow a single Python-literal style line as loose fallback
            try:
                obj = ast.literal_eval(ln)
            except Exception:
                snippet = (ln[:120] + "...") if len(ln) > 120 else ln
                sys.exit(f"NDJSON parse error on line {idx+1}: {e.msg}. Line: {snippet}")
        records.append(obj)
    if not records:
        sys.exit("NDJSON input contained no records.")
    return records

def normalize_record(item, index):
    # Validate required keys
    for k in ("base_cluster", "other_cluster", "group", "outlier_id"):
        if k not in item:
            sys.exit(f"Missing key '{k}' in item index {index}.")

    base_cluster = item["base_cluster"]
    other_cluster = item["other_cluster"]
    group = item["group"]
    outlier_id = item["outlier_id"]

    if not isinstance(base_cluster, (list, tuple)) \
       or not isinstance(other_cluster, (list, tuple)) \
       or not isinstance(group, (list, tuple)):
        sys.exit(f"'base_cluster', 'other_cluster', and 'group' must be lists (item {index}).")

    group_strs = [str(x) for x in group]
    outlier_id_val = coerce_int_if_numeric(outlier_id)

    return base_cluster, other_cluster, group_strs, outlier_id_val

In [ ]:
# ap = argparse.ArgumentParser()
# ap.add_argument("input", help="Path to JSON file (array). Use '-' for stdin.")
# ap.add_argument("-o", "--output", default="converted.csv",
#                 help="Output CSV path. Use '-' for stdout. Default: converted.csv")
# ap.add_argument("--seed", type=int, default=None,
#                 help="Base seed for deterministic shuffling of 'options'. If omitted, uses non-deterministic randomness.")
# args = ap.parse_args()

# # Read JSON
# if args.input == "-":
#     raw = sys.stdin.read()
# else:
#     with open(args.input, "r", encoding="utf-8") as f:
#         raw = f.read()
raw = "easy_dataset_mid.json"
output = "easy_dataset_mid.csv"
try:
    data = json.loads(raw)
except json.JSONDecodeError as e:
    sys.exit(f"Failed to parse JSON: {e}")

if not isinstance(data, list):
    sys.exit("Input JSON must be an array of objects.")

# Prepare RNG
sys_rng = random.SystemRandom()
# base_seed = args.seed if args.seed is not None else sys_rng.randrange(1 << 30)
base_seed = 42

try:

    data = parse_as_json_array(raw, loose=args.loose)
except json.JSONDecodeError as e:
    # Give an actionable message with a short snippet
    snippet = (raw[:200] + "...") if len(raw) > 200 else raw
    sys.exit(f"Failed to parse JSON: {e.msg} at pos {e.pos}. "
                f"Make sure the input is a valid JSON array. Snippet:\n{snippet}")

if not isinstance(data, list):
    sys.exit("Parsed input is not a list of objects.")

# Build rows
rows = []
for i, item in enumerate(data):
    base_cluster, other_cluster, group_strs, outlier_id_val = normalize_record(item, i)

    rng = random.Random(base_seed + i)  # deterministic per-row
    options = group_strs[:]
    rng.shuffle(options)

    rows.append({
        "base_cluster": str(list(base_cluster)),
        "other_cluster": str(list(other_cluster)),
        "group": str(list(group_strs)),
        "outlier_id": outlier_id_val,
        "options": str(list(options)),
    })

# Write CSV
fieldnames = ["base_cluster", "other_cluster", "group", "outlier_id", "options"]
if args.output == "-":
    writer = csv.DictWriter(sys.stdout, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
else:
    with open(args.output, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

SystemExit: Failed to parse JSON: Expecting value: line 1 column 1 (char 0)

/ssd2/aliceliu/surprise_sae/planscape/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
